## Read Data from Bronze Layer

In [0]:
%run ./test

In [0]:
def load_data(table):
  return spark.read.table(table)


## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_load_data'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Get Star Players from Common Player Info

By filtering the greatest 75th anniversay team players

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col

def get_star_players(common_player_info_df: DataFrame):
    return common_player_info_df.where(col("greatest_75_flag") == "Y").withColumn("player_id", col("person_id")).drop("person_id").select("player_id", "first_name", "last_name", "display_first_last", "position", "from_year", "to_year")

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_get_star_players'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Game with Star Absence

Get the games that the star players are off

In [0]:
def process_game_with_star_absence(star_players_df: DataFrame, inactive_players_df: DataFrame):
    return star_players_df.join(inactive_players_df, on="player_id")
    # .select("game_id", "player_id", "team_id", "team_name")


## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_process_game_with_star_absence'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Game with Winning and Losing

Only consider the regular season games

In [0]:
def procee_game_with_winning_and_losing(game_with_star_absence_df: DataFrame, game_df: DataFrame):
    return game_with_star_absence_df.join(game_df, on="game_id").where(col("season_type") == "Regular Season")

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_procee_game_with_winning_and_losing'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Game 

In [0]:
INACTIVE_PLAYERS_TABLE = "nba_insights.bronze.inactive_players"
COMMON_PLAYER_INFO_TABLE = "nba_insights.bronze.common_player_info"
GAME_TABLE = "nba_insights.bronze.games"

inactive_players_df = load_data(INACTIVE_PLAYERS_TABLE)
common_player_info_df = load_data(COMMON_PLAYER_INFO_TABLE)
game_df = load_data(GAME_TABLE)

def process_game(common_player_info_df: DataFrame, game_df: DataFrame, inactive_players_df: DataFrame):
    star_df = common_player_info_df.transform(get_star_players)
    game_with_star_absence_df = star_df.transform(process_game_with_star_absence, inactive_players_df)
    game_stats_with_star_absence_df = game_with_star_absence_df.transform(procee_game_with_winning_and_losing, game_df)\
        .withColumn("star_team_id", col("team_id"))\
            .drop("team_id")\
                .select("game_id", "star_team_id", "team_id_home", "wl_home", "team_id_away", "wl_away", "player_id")

    return game_stats_with_star_absence_df


INACTIVE_PLAYERS_TABLE = "nba_insights.bronze.inactive_players"
COMMON_PLAYER_INFO_TABLE = "nba_insights.bronze.common_player_info"
GAME_TABLE = "nba_insights.bronze.game"

game_stats_with_star_absence_df = process_game(common_player_info_df, game_df, inactive_players_df)
display(game_stats_with_star_absence_df)

# process_game(common_player_info_df, game_df, inactive_players_df).write.mode("overwrite").saveAsTable(nba_insights.silver.games_stats_with_absence_stars)

## E2E Test

In [0]:
# E2E Test

## Persist Silver Layer

In [0]:
common_player_info_df = load_data(COMMON_PLAYER_INFO_TABLE)
star_players_df = get_star_players(common_player_info_df)

star_players_df = common_player_info_df.transform(get_star_players)
star_players_df.write.mode("overwrite").saveAsTable("nba_insights.silver.star_players")

game_stats_with_star_absence_df.write.mode("overwrite").saveAsTable("nba_insights.silver.games_stats_with_stars_absence")